## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 关键：必须在创建任何 Tensor/Variable 之前关闭 Eager
tf.compat.v1.disable_eager_execution()

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    x = x / 255.0
    x_test = x_test / 255.0
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
# 定义网络参数（Graph 模式变量）
hidden = 100
in_dim = 28 * 28
out_dim = 10

W1 = tf.Variable(tf.random.normal([in_dim, hidden], stddev=0.1, dtype=tf.float32), name="W1")
b1 = tf.Variable(tf.zeros([hidden], dtype=tf.float32), name="b1")
W2 = tf.Variable(tf.random.normal([hidden, out_dim], stddev=0.1, dtype=tf.float32), name="W2")
b2 = tf.Variable(tf.zeros([out_dim], dtype=tf.float32), name="b2")

def forward(x):
    x = tf.reshape(x, [-1, 28 * 28])
    h1 = tf.nn.relu(tf.matmul(x, W1) + b1)
    logits = tf.matmul(h1, W2) + b2
    return logits


## 计算 loss

In [4]:
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels
        )
    )

def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1, output_type=labels.dtype)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

def train_one_step(optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = forward(x)
        loss = compute_loss(logits, y)

    trainable_vars = [W1, b1, W2, b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

def test(x, y):
    logits = forward(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy


## 实际训练

In [5]:
# 构图：placeholder -> forward -> loss/acc -> train_op
x_ph = tf.compat.v1.placeholder(tf.float32, shape=[None, 28, 28], name="x")
y_ph = tf.compat.v1.placeholder(tf.int64, shape=[None], name="y")

logits = forward(x_ph)

loss = tf.reduce_mean(
    tf.nn.sparse_softmax_cross_entropy_with_logits(logits=logits, labels=y_ph)
)

pred = tf.argmax(logits, axis=1, output_type=tf.int64)
accuracy = tf.reduce_mean(tf.cast(tf.equal(pred, y_ph), tf.float32))

train_op = tf.compat.v1.train.AdamOptimizer(learning_rate=1e-3).minimize(loss)

# 训练与测试（Session）
train_data, test_data = mnist_dataset()
x_train = train_data[0].astype(np.float32)
y_train = train_data[1].astype(np.int64)
x_test = test_data[0].astype(np.float32)
y_test = test_data[1].astype(np.int64)

with tf.compat.v1.Session() as sess:
    sess.run(tf.compat.v1.global_variables_initializer())

    for epoch in range(50):
        l, a, _ = sess.run(
            [loss, accuracy, train_op],
            feed_dict={x_ph: x_train, y_ph: y_train},
        )
        print("epoch", epoch, ": loss", float(l), "; accuracy", float(a))

    l_test, a_test = sess.run(
        [loss, accuracy],
        feed_dict={x_ph: x_test, y_ph: y_test},
    )
    print("test loss", float(l_test), "; accuracy", float(a_test))

epoch 0 : loss 2.5813822746276855 ; accuracy 0.12150000035762787
epoch 1 : loss 2.415104866027832 ; accuracy 0.15541666746139526
epoch 2 : loss 2.2823221683502197 ; accuracy 0.20784999430179596
epoch 3 : loss 2.173877239227295 ; accuracy 0.2589166760444641
epoch 4 : loss 2.0826680660247803 ; accuracy 0.30674999952316284
epoch 5 : loss 2.0031869411468506 ; accuracy 0.36434999108314514
epoch 6 : loss 1.9312512874603271 ; accuracy 0.4229666590690613
epoch 7 : loss 1.8637783527374268 ; accuracy 0.46131667494773865
epoch 8 : loss 1.7985143661499023 ; accuracy 0.48820000886917114
epoch 9 : loss 1.733949899673462 ; accuracy 0.5119166374206543
epoch 10 : loss 1.6692885160446167 ; accuracy 0.5387166738510132
epoch 11 : loss 1.604391098022461 ; accuracy 0.5664499998092651
epoch 12 : loss 1.5396288633346558 ; accuracy 0.5939666628837585
epoch 13 : loss 1.4757134914398193 ; accuracy 0.6216166615486145
epoch 14 : loss 1.413480520248413 ; accuracy 0.6461166739463806
epoch 15 : loss 1.353623390197754